# Week 11

Two Hybrid recommender models using Parallel combination strategy 
one from SVD CF combined with content based from week 10.   
The other SVD CF combined with content based model with gemma3 generated descriptions

In [65]:
%pip install ollama

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [47]:
# load imports
import pandas as pd
from pathlib import Path
import ollama

# load user-item interactions
user_item_matrix = pd.read_csv('csv_files/user_item_matrix.csv', index_col=0)
# load train and test
train_df = pd.read_csv('csv_files/preprocessed_train_video_games.csv')
test_df = pd.read_csv('csv_files/preprocessed_test_video_games.csv')
metadata_df = pd.read_parquet('datasets/meta_video_games.parquet')


You can make the Option 2 more effcient by generating descriptions only
for the top-k (with a large enough k) items recommended by the collaborative filtering model, instead of all items in the dataset. Then, your hybrid approach
will re-rank the top-k items

### 1) Collaborative Filtering Base
1. Use your strongest CF setup from previous weeks (e.g., KNN or SVD). 
1b. use the saved SVD model that was pickled
2. Generate predicted scores for unrated (user, item) pairs.
3. Keep full candidate ranking or at least top-K candidates per user.

In [48]:
# load best svd model (user-item matrix with predictions)
u_i_matrix_svd = pd.read_csv("csv_files/user_item_matrix_svd.csv", index_col=0)
u_i_matrix_svd.info()
# check for null cells
u_i_matrix_svd.isnull().sum().sum()

# load unobserved predictions svd
unobserved_preds_svd = pd.read_csv("csv_files/unobserved_predictions_svd.csv", index_col=0)
# sort by rating in descending order
unobserved_preds_svd = unobserved_preds_svd.sort_values("svd_rating", ascending=False)
unobserved_preds_svd.head()

<class 'pandas.core.frame.DataFrame'>
Index: 1389 entries, AE25ZDXYBK3LHKCZ7XUODANPME4A to AHZYXDJ3HNLKS2E73VOSNIZZJT4Q
Columns: 932 entries, B00000JRSB to B0C5K4M7WJ
dtypes: float64(932)
memory usage: 9.9+ MB


,item_id,svd_rating
user_id,,
AELD7NXSVVFKTNFB3I673NGJVSWA,B00J4Y6L2Y,5.0
AF52HVGBBCP3JAS6B7B3I4AVIWRQ,B017W1771Y,5.0
AHMG3HUA4K6H475VQGJF2QAOTFTQ,B0BDWVBWC9,5.0
AF52HVGBBCP3JAS6B7B3I4AVIWRQ,B004QEV0MI,5.0
AGP7FLSJ7BHYFVZY3FRKVVDBH47Q,B007CSF3GO,5.0


#### DO a SUBSET for easier time matching common item between all the sets and lower compuational cost
how much to filter by out of the 927? 


### 2) Content-Based Base (Original Metadata)
1. Implement same CB representation as Week 10.
1b. rerun week 10 and save the content based model as pickle file
2. prompt an agent and ask how I can create an unobserved df frame from the predicitions in week 10

In [49]:
# Build content-based predictions for unobserved pairs (for hybrid in Week 11)

import numpy as np
import pandas as pd
import pickle

# 1) Load Week 10 trained artifacts
with open("models/best_content_based_model.pkl", "rb") as f:
    cb_artifacts = pickle.load(f)

user_id_to_row = cb_artifacts["user_id_to_row"]
item_id_to_row = cb_artifacts["item_id_to_row"]
user_matrix = cb_artifacts["user_matrix"]   # sparse
item_matrix = cb_artifacts["item_matrix"]   # sparse 


# 2) Load the same unobserved pair list used by SVD
# If your file has an index column, this still works safely.
unobs_pairs = unobserved_preds_svd.copy()

# this is because the csv headers are not read in properly
if "user_id" not in unobs_pairs.columns:
    unobs_pairs = unobs_pairs.reset_index().rename(columns={"index": "user_id"})

required = {"user_id", "item_id"}
missing = required.difference(unobs_pairs.columns)
if missing:
    raise ValueError(f"Missing required columns in unobserved file: {missing}")

unobs_pairs["user_id"] = unobs_pairs["user_id"].astype(str)
unobs_pairs["item_id"] = unobs_pairs["item_id"].astype(str)

# 3) Keep only pairs that exist in CB mappings
pairs = unobs_pairs[
    unobs_pairs["user_id"].isin(user_id_to_row) &
    unobs_pairs["item_id"].isin(item_id_to_row)
].copy()


# After the .isin() filter (NaNs are already gone), force int explicitly
pairs["u_row"] = pairs["user_id"].map(user_id_to_row).astype(int)
pairs["i_row"] = pairs["item_id"].map(item_id_to_row).astype(int)


print("user_matrix shape:", user_matrix.shape)
print("item_matrix shape:", item_matrix.shape)

print("\nu_row stats:")
print(pairs["u_row"].describe())
print("Negative u_rows:", (pairs["u_row"] < 0).sum())
print("u_row >= user_matrix.shape[0]:", (pairs["u_row"] >= user_matrix.shape[0]).sum())

print("\ni_row stats:")
print(pairs["i_row"].describe())
print("Negative i_rows:", (pairs["i_row"] < 0).sum())
print("i_row >= item_matrix.shape[0]:", (pairs["i_row"] >= item_matrix.shape[0]).sum())



user_matrix shape: (1389, 8056)
item_matrix shape: (927, 8056)

u_row stats:
count    1.261123e+06
mean     6.951193e+02
std      4.010045e+02
min      0.000000e+00
25%      3.480000e+02
50%      6.960000e+02
75%      1.043000e+03
max      1.388000e+03
Name: u_row, dtype: float64
Negative u_rows: 0
u_row >= user_matrix.shape[0]: 0

i_row stats:
count    1.261123e+06
mean     4.628827e+02
std      2.678035e+02
min      0.000000e+00
25%      2.310000e+02
50%      4.630000e+02
75%      6.950000e+02
max      9.260000e+02
Name: i_row, dtype: float64
Negative i_rows: 0
i_row >= item_matrix.shape[0]: 0


In [50]:
from typing import Iterable

def score_unobserved_pairs_cosine(unobs_pairs: pd.DataFrame, user_matrix, item_matrix, user_id_to_row: dict, item_id_to_row: dict, allowed_item_ids: Iterable | None = None, user_col: str = "user_id", item_col: str = "item_id", score_col: str = "cb_score_cosine", clip_min: float = 0.0, clip_max: float = 1.0, chunk_size: int = 50000, copy_df: bool = True) -> pd.DataFrame:
    """Score unobserved user-item pairs via cosine similarity from aligned user/item sparse matrices."""
    if user_col not in unobs_pairs.columns or item_col not in unobs_pairs.columns:
        raise KeyError(f"Expected columns '{user_col}' and '{item_col}' in unobs_pairs")

    out_df = unobs_pairs.copy(deep=True) if copy_df else unobs_pairs
    out_df[user_col] = out_df[user_col].astype(str)
    out_df[item_col] = out_df[item_col].astype(str)

    if allowed_item_ids is not None:
        allowed_set = set(str(x) for x in allowed_item_ids)
        out_df = out_df[out_df[item_col].isin(allowed_set)].copy()

    out_df = out_df[out_df[user_col].isin(user_id_to_row) & out_df[item_col].isin(item_id_to_row)].copy()
    if out_df.empty:
        return out_df[[user_col, item_col]].assign(**{score_col: np.array([], dtype=float)})

    out_df["u_row"] = out_df[user_col].map(user_id_to_row).astype(int)
    out_df["i_row"] = out_df[item_col].map(item_id_to_row).astype(int)

    if out_df["u_row"].max() >= user_matrix.shape[0] or out_df["i_row"].max() >= item_matrix.shape[0]:
        raise ValueError("Mapped row indices exceed user_matrix or item_matrix dimensions")

    u_idx = out_df["u_row"].to_numpy(dtype=np.int64)
    i_idx = out_df["i_row"].to_numpy(dtype=np.int64)
    user_norms = np.sqrt(np.asarray(user_matrix.multiply(user_matrix).sum(axis=1)).ravel())
    item_norms = np.sqrt(np.asarray(item_matrix.multiply(item_matrix).sum(axis=1)).ravel())

    cos = np.empty(len(out_df), dtype=np.float32)
    for start in range(0, len(out_df), chunk_size):
        end = min(start + chunk_size, len(out_df))
        uu = u_idx[start:end]
        ii = i_idx[start:end]
        dot_ui = np.asarray(user_matrix[uu].multiply(item_matrix[ii]).sum(axis=1)).ravel()
        den = user_norms[uu] * item_norms[ii]
        c = np.divide(dot_ui, den, out=np.zeros_like(dot_ui, dtype=float), where=den > 0)
        cos[start:end] = np.clip(c, clip_min, clip_max)

    scored_df = out_df[[user_col, item_col]].copy()
    scored_df[score_col] = cos
    return scored_df

In [51]:
# Generate CB scores for all mappable unobserved pairs and save output (reusable helper)

cb_out = score_unobserved_pairs_cosine(
    unobs_pairs=pairs,
    user_matrix=user_matrix,
    item_matrix=item_matrix,
    user_id_to_row=user_id_to_row,
    item_id_to_row=item_id_to_row,
    score_col="cb_score_cosine",
    chunk_size=50000,
)
cb_out["cb_rating"] = 1.0 + 4.0 * cb_out["cb_score_cosine"]

out_path = 'csv_files/unobserved_predictions_cb.csv'
cb_out.to_csv(out_path, index=False) # index means numbering the rows

print('Saved:', out_path)
print('Rows saved:', len(cb_out))
cb_out.head()

Saved: csv_files/unobserved_predictions_cb.csv
Rows saved: 1261123


,user_id,item_id,cb_score_cosine,cb_rating
0,AELD7NXSVVFKTNFB3I673NGJVSWA,B00J4Y6L2Y,0.204246,1.816986
1,AF52HVGBBCP3JAS6B7B3I4AVIWRQ,B017W1771Y,0.077754,1.311018
2,AHMG3HUA4K6H475VQGJF2QAOTFTQ,B0BDWVBWC9,0.067831,1.271324
3,AF52HVGBBCP3JAS6B7B3I4AVIWRQ,B004QEV0MI,0.155047,1.620187
4,AGP7FLSJ7BHYFVZY3FRKVVDBH47Q,B007CSF3GO,0.180029,1.720118


### 3) Content-Based with Gemma3 (LLM Metadata)
1. For each item title, generate a short, structured description using Gemma via Ollama.
2. Create text features from generated descriptions (same vectorization pipeline used in CB A where possible).
3. Compute CB scores for unrated items.


In [52]:
# Build SVD candidate pool using per-user top-k (candidate generation stage)
k_per_user = 10

unobs_svd = unobserved_preds_svd.copy()
if "user_id" not in unobs_svd.columns:
    unobs_svd = unobs_svd.reset_index()

required_cols = {"user_id", "item_id", "svd_rating"}
missing = required_cols.difference(unobs_svd.columns)
if missing:
    raise ValueError(f"Missing required columns in unobserved SVD predictions: {missing}")

unobs_svd["user_id"] = unobs_svd["user_id"].astype(str)
unobs_svd["item_id"] = unobs_svd["item_id"].astype(str)

topk_per_user_svd = (unobs_svd.sort_values(["user_id", "svd_rating"], ascending=[True, False]).groupby("user_id", as_index=False, group_keys=False).head(k_per_user).reset_index(drop=True))
# filter to unique items across all users, keeping the highest rating for each item
candidate_items_svd = (topk_per_user_svd.groupby("item_id", as_index=False)["svd_rating"].max().sort_values("svd_rating", ascending=False).reset_index(drop=True))

# remove duplicates
candidate_item_ids_svd = set(candidate_items_svd["item_id"].astype(str))

print(f"Users in SVD predictions: {topk_per_user_svd['user_id'].nunique()}")
print(f"Per-user top-k: {k_per_user}")
print(f"Rows kept after per-user top-k: {len(topk_per_user_svd):,}")
print(f"Unique candidate items: {len(candidate_item_ids_svd):,}")

candidate_items_svd.head(10)

Users in SVD predictions: 1389
Per-user top-k: 10
Rows kept after per-user top-k: 13,890
Unique candidate items: 546


,item_id,svd_rating
0,B00000JRSB,5.0
1,B00MB1I3FU,5.0
2,B00RU75I2G,5.0
3,B00R0ZS9YW,5.0
4,B00QO4NAOO,5.0
5,B00PQ1OQ4Y,5.0
6,B00PBIGBOU,5.0
7,B00OGNV5HY,5.0
8,B00OBZNI0O,5.0
9,B00NOD0OTW,5.0


filter by per-user top-k  (1389 users)  
get the top 10 highest rated item per user  
then combine all the top 10 per user lists  
then remove duplicates  
items went from 927 to 546

#### so many trade offs and decisions
how long should the description be?
what should it included?
how detailed? ect.

In [53]:
# filter by item ids in test set
metadata_df = metadata_df[metadata_df['item_id'].isin(test_df['item_id'])]
metadata_df.head()
print(len(metadata_df), len(test_df["item_id"].unique()))

927 927


Be able to change the model and try different LLMs

In [108]:
model_name = "gemma3" 

In [109]:
# 3) Content-Based Base (LLM Metadata) - Gemma description generation on SVD candidates only
from concurrent.futures import ThreadPoolExecutor, as_completed

if "candidate_item_ids_svd" not in globals():
    raise RuntimeError("Run the per-user top-k candidate cell before this cell.")

metadata_llm_df = metadata_df.copy()
metadata_llm_df["item_id"] = metadata_llm_df["item_id"].astype(str)
metadata_llm_df["title"] = metadata_llm_df["title"].fillna("").astype(str).str.strip()
metadata_llm_df = metadata_llm_df[metadata_llm_df["title"] != ""].copy()

# Only keep item metadata for CF candidate items
metadata_llm_df = metadata_llm_df[metadata_llm_df["item_id"].isin(candidate_item_ids_svd)].copy()

items_df = (metadata_llm_df[["item_id", "title"]].drop_duplicates(subset=["item_id"]).reset_index(drop=True))

if model_name == "qwen3:4b": # need this because colons can't be in file names
    cache_path = Path(f"csv_files/qwen3_llm_item_descriptions.csv")
else:
    cache_path = Path(f"csv_files/{model_name}_llm_item_descriptions.csv")

deterministic_options = {"temperature": 0.0,"top_p": 1.0,"seed": 42,}

# Keep worker count small to avoid overloading local Ollama
use_parallel = True
max_workers = 3

def generate_short_description(title_text: str) -> str:
    prompt = (
        "Write 1-2 concise sentences describing this video game title for recommender metadata. "
        "Focus on likely genre, gameplay style, and intended audience. "
        "Do not use bullet points. Title: " + title_text
    )
    response = ollama.chat(
        model=model_name,
        messages=[
            {"role": "system", "content": "You generate concise, neutral recommendation metadata."},
            {"role": "user", "content": prompt},
        ],
        options=deterministic_options,
    )
    return response["message"]["content"].strip()

def generate_one_row(row):
    item_id = str(row.item_id)
    title_text = row.title
    try:
        desc = generate_short_description(title_text)
    except Exception as exc:
        desc = ""
        print(f"[WARN] generation failed for item_id={item_id}: {exc}")

    return {
        "item_id": item_id,
        "title": title_text,
        "llm_description": desc,
    }

if cache_path.exists():
    cached_df = pd.read_csv(cache_path, dtype={"item_id": str})
    cached_df = cached_df[["item_id", "title", "llm_description"]].drop_duplicates(subset=["item_id"])
else:
    cached_df = pd.DataFrame(columns=["item_id", "title", "llm_description"])

items_df["item_id"] = items_df["item_id"].astype(str)
done_item_ids = set(cached_df["item_id"].astype(str))
todo_df = items_df[~items_df["item_id"].isin(done_item_ids)].copy()

print(f"Candidate items from SVD top-k: {len(items_df)}")
print(f"Already cached: {len(cached_df)}")
print(f"To generate now: {len(todo_df)}")

new_rows = []
todo_rows = list(todo_df.itertuples(index=False))

if use_parallel and len(todo_rows) > 0:
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(generate_one_row, row) for row in todo_rows]
        for i, future in enumerate(as_completed(futures), start=1):
            new_rows.append(future.result())
            if i % 25 == 0 or i == len(todo_rows):
                print(f"Generated {i}/{len(todo_rows)} new descriptions...")
else:
    for i, row in enumerate(todo_rows, start=1):
        new_rows.append(generate_one_row(row))
        if i % 25 == 0 or i == len(todo_rows):
            print(f"Generated {i}/{len(todo_rows)} new descriptions...")

if new_rows:
    new_df = pd.DataFrame(new_rows)
    final_df = pd.concat([cached_df, new_df], ignore_index=True)
else:
    final_df = cached_df.copy()

final_df = final_df.drop_duplicates(subset=["item_id"], keep="last")
final_df = final_df.sort_values("item_id").reset_index(drop=True)
final_df.to_csv(cache_path, index=False)

llm_descriptions_df = final_df.copy()
print(f"Saved cache to: {cache_path}")
print(f"Cached descriptions total: {len(llm_descriptions_df)}")
llm_descriptions_df.head()

Candidate items from SVD top-k: 543
Already cached: 543
To generate now: 0
Saved cache to: csv_files\gemma3_llm_item_descriptions.csv
Cached descriptions total: 543


,item_id,title,llm_description
0,B00000JRSB,Final Fantasy VII - PlayStation,Final Fantasy VII – PlayStation is a classic r...
1,B00001X50M,Metal Gear Solid,Metal Gear Solid is a stealth-action game wher...
2,B00001XDUB,Silent Hill,Silent Hill is a psychological horror game tha...
3,B0000296O5,Final Fantasy VIII,Final Fantasy VIII is a classic JRPG offering ...
4,B00004Y57G,Final Fantasy IX,Final Fantasy IX is a classic JRPG offering a ...


Another round of remove duplicates change rows from 546 -> 543

### Test LLM Content Based Recommender
Preprocess text and go through content based TF-IDF pipeline. 

In [110]:
import evals_helper as eh

# text preprocessing
llm_descriptions_df_processed = eh.preprocess_text_column(llm_descriptions_df, "llm_description")
llm_descriptions_df_processed.head()
# tfidf vectorization on the llm descriptions
tfidf_vectorizer, tfidf_matrix = eh.tfidf_vectorizing(llm_descriptions_df_processed, 'llm_description')

feature_names = tfidf_vectorizer.get_feature_names_out()
item_ids = llm_descriptions_df_processed['item_id'].tolist()
print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'Number of features: {len(feature_names)}')
print(f'Number of unique item ids: {len(set(item_ids))}')

TF-IDF matrix shape: (543, 1898)
Number of features: 1898
Number of unique item ids: 543


In [111]:
# represent users in TF-IDF space by averaging vectors of interacted items
train_interactions = train_df[["user_id", "item_id", "rating"]].copy()
train_interactions["item_id"] = train_interactions["item_id"].astype(str)
user_profiles = eh.build_user_profiles(train_interactions, tfidf_matrix, item_ids)

# Build mappings aligned to the row order of user_profiles and tfidf_matrix
filtered_interactions = train_interactions[train_interactions["item_id"].isin(set(item_ids))].copy()
profile_user_ids = filtered_interactions["user_id"].drop_duplicates().tolist()
user_profile_id_to_row = {str(user_id): idx for idx, user_id in enumerate(profile_user_ids)}
item_id_to_row_llm = {str(item_id): idx for idx, item_id in enumerate(item_ids)}

print(f'User profiles shape: {user_profiles.shape}')
print(f'Mapped users in profiles: {len(user_profile_id_to_row)}')
print(f'Mapped LLM items: {len(item_id_to_row_llm)}')

User profiles shape: (1389, 1898)
Mapped users in profiles: 1389
Mapped LLM items: 543


In [112]:
# score unobserved pairs using cosine similarity between user profiles and LLM item vectors
cb_out_llm = score_unobserved_pairs_cosine(
    unobs_pairs=unobs_pairs,
    user_matrix=user_profiles,
    item_matrix=tfidf_matrix,
    user_id_to_row=user_profile_id_to_row,
    item_id_to_row=item_id_to_row_llm,
    allowed_item_ids=llm_descriptions_df_processed["item_id"],
    score_col="cb_score_cosine",
    chunk_size=50000,
)

cb_out_llm["cb_rating"] = 1.0 + 4.0 * cb_out_llm["cb_score_cosine"]

print(f'Rows scored ( {model_name} LLM CB): {len(cb_out_llm)}')
cb_out_llm.head()

Rows scored ( gemma3 LLM CB): 736876


,user_id,item_id,cb_score_cosine,cb_rating
0,AELD7NXSVVFKTNFB3I673NGJVSWA,B00J4Y6L2Y,0.097889,1.391558
1,AF52HVGBBCP3JAS6B7B3I4AVIWRQ,B017W1771Y,0.084572,1.338286
2,AHMG3HUA4K6H475VQGJF2QAOTFTQ,B0BDWVBWC9,0.027473,1.109894
3,AF52HVGBBCP3JAS6B7B3I4AVIWRQ,B004QEV0MI,0.052948,1.211794
4,AGP7FLSJ7BHYFVZY3FRKVVDBH47Q,B007CSF3GO,0.169584,1.678337


In [113]:
# evaluate just LLM rmse and mae using the helper function

_ , map = eh.calculate_map_at_k(test_df, cb_out_llm, prediction_col="cb_rating", k=10)
hitrate = eh.calculate_hit_rate(test_df=test_df,recommendations_df=cb_out_llm, prediction_col="cb_rating", k=10)
precision = eh.calculate_precision_at_k(test_df=test_df, recommendations_df=cb_out_llm, prediction_col="cb_rating", k=10)
mrr = eh.calculate_mrr_at_k(test_df=test_df, recommendations_df=cb_out_llm, prediction_col="cb_rating", k=10)
coverage = eh.calculate_coverage_at_k(test_df=test_df, recommendations_df=cb_out_llm, prediction_col="cb_rating", k=10)
print(f"{model_name} LLM CB Hit Rate@10: {hitrate:.4f}")
print(f"{model_name} LLM CB MAP@10: {map:.4f}")
print(f"{model_name} LLM CB Precision@10: {precision:.4f}")
print(f"{model_name} LLM CB MRR@10: {mrr:.4f}")
print(f"{model_name} LLM CB Coverage@10: {coverage:.4f}")



gemma3 LLM CB Hit Rate@10: 0.1821
gemma3 LLM CB MAP@10: 0.0193
gemma3 LLM CB Precision@10: 0.0209
gemma3 LLM CB MRR@10: 0.0668
gemma3 LLM CB Coverage@10: 0.4865


REMEMBER, less rows because we are only doing top-10 per user

### Combining Strategies

Combine CF and CB with normal strategies 

LLM is supposed to rerank the CF 543 items. How?

Weighted Sum
This method is used to combine ratings (e.g., predicted star ratings)
.
How it works: You multiply the output of each model by an assigned weight (α,β) and sum them up: R 
H
​
 =αR 
A
​
 +βR 
B
​
 
.

In [114]:
# implement reusable weighted-sum hybrid function to combine CF and CB methods
def combine_predictions_weighted_sum(df1: pd.DataFrame, df2: pd.DataFrame, on_cols: list, rating_col1: str, rating_col2: str, alpha: float, beta: float, out_rating_col: str = "hybrid_score") -> pd.DataFrame:
    """Combine two sets of predictions via a weighted sum."""
    if not np.isclose(alpha + beta, 1.0):
        raise ValueError("Weights must sum to 1.0")

    # Helper to ensure required columns are present, handling common index/CSV issues.
    def _normalize_merge_frame(df: pd.DataFrame, required_cols: list) -> pd.DataFrame:
        out = df.copy()
        if all(col in out.columns for col in required_cols):
            return out

        # If key columns are in index, bring them back as columns.
        if out.index.name in required_cols or any(name in required_cols for name in getattr(out.index, "names", []) if name is not None):
            out = out.reset_index()
            if all(col in out.columns for col in required_cols):
                return out

        # Handle common CSV index artifact.
        if "index" in out.columns and "user_id" in required_cols and "user_id" not in out.columns:
            out = out.rename(columns={"index": "user_id"})
            if all(col in out.columns for col in required_cols):
                return out

        missing = [col for col in required_cols if col not in out.columns]
        raise KeyError(f"Missing required columns {missing}. Available columns: {list(out.columns)}")

    req1 = on_cols + [rating_col1]
    req2 = on_cols + [rating_col2]
    df1n = _normalize_merge_frame(df1, req1)
    df2n = _normalize_merge_frame(df2, req2)

    # Normalize key types to avoid empty merges caused by str/int mismatch.
    for col in on_cols:
        df1n[col] = df1n[col].astype(str)
        df2n[col] = df2n[col].astype(str)

    merged = pd.merge(df1n[req1], df2n[req2], on=on_cols, how="inner")
    merged[out_rating_col] = alpha * merged[rating_col1] + beta * merged[rating_col2]
    return merged[on_cols + [out_rating_col]]

In [115]:
# combine non filtered svd and regular cb (from week 10)
# what columns make sense to combine? the cbs have two different columns
# one for cosine similarity and one for rating scale 1-5 
# svd only has its rating scale 1-5, so we should use that one for cb as well
hybrid_svd_cb = combine_predictions_weighted_sum(
    # what is the purpose of this renaming? 
    # it's to ensure that the rating columns have consistent names for the weighted sum calculation.
    # I want the columns to be refered to as cb_rating and svd rating in the combine_predictions_weighted_sum function, so I rename them here before passing to the function.
    df1=unobserved_preds_svd,
    df2=cb_out,
    on_cols=["user_id", "item_id"],
    rating_col1="svd_rating",
    rating_col2="cb_rating",
    alpha=0.3,
    beta=0.7,
    out_rating_col="hybrid_svd_cb_rating"
)
print(f"Combined SVD and regular CB ratings: {len(hybrid_svd_cb)} rows")
hybrid_svd_cb.head()

Combined SVD and regular CB ratings: 1261123 rows


,user_id,item_id,hybrid_svd_cb_rating
0,AELD7NXSVVFKTNFB3I673NGJVSWA,B00J4Y6L2Y,2.771890
1,AF52HVGBBCP3JAS6B7B3I4AVIWRQ,B017W1771Y,2.417713
2,AHMG3HUA4K6H475VQGJF2QAOTFTQ,B0BDWVBWC9,2.389927
3,AF52HVGBBCP3JAS6B7B3I4AVIWRQ,B004QEV0MI,2.634131
4,AGP7FLSJ7BHYFVZY3FRKVVDBH47Q,B007CSF3GO,2.704082


In [116]:
# Combine SVD and LLM CB predictions using the weighted sum function
# Recall that we use top-k svd candidates, so we need to use candidate_items_svd
cb_llm_subset = cb_out_llm[cb_out_llm["item_id"].isin(candidate_item_ids_svd)].copy() 
# The cb llm is already based on the candidates
# This is redundant safety check, but ensures we only combine on the intended candidate set

hybrid_svd_cb_llm = combine_predictions_weighted_sum(
    df1=topk_per_user_svd,
    df2=cb_llm_subset,
    on_cols=["user_id", "item_id"],
    rating_col1="svd_rating",
    rating_col2="cb_rating",
    alpha=0.8,
    beta=0.2,
    out_rating_col="hybrid_svd_cb_llm_rating"
)
print(f"Hybrid SVD + {model_name} LLM CB rows: {len(hybrid_svd_cb_llm)}")
hybrid_svd_cb_llm.head()

Hybrid SVD + gemma3 LLM CB rows: 13826


,user_id,item_id,hybrid_svd_cb_llm_rating
0,AE25ZDXYBK3LHKCZ7XUODANPME4A,B000FQ9QVI,2.316706
1,AE25ZDXYBK3LHKCZ7XUODANPME4A,B004HILZUU,2.406999
2,AE25ZDXYBK3LHKCZ7XUODANPME4A,B00000JRSB,2.331270
3,AE25ZDXYBK3LHKCZ7XUODANPME4A,B00001X50M,2.261447
4,AE25ZDXYBK3LHKCZ7XUODANPME4A,B0002A6CQ4,2.245162


Reciprocal Rank Fusion (RRF)
This method combines rankings (not raw scores).
How it works: for each model ranking, assign rank 1 to the best item, then sum reciprocal rank terms per item:
RRF score = sum(1 / (k + rank_m)) across models m, where k is typically 60.

In [117]:
def combine_predictions_rrf(df1: pd.DataFrame, df2: pd.DataFrame, on_cols: list, score_col1: str, score_col2: str, k: int = 60, out_rating_col: str = "hybrid_rrf_score") -> pd.DataFrame:
    """Combine two ranked prediction lists using Reciprocal Rank Fusion (RRF)."""
    if k <= 0:
        raise ValueError("k must be > 0 for RRF")

    def _normalize_merge_frame(df: pd.DataFrame, required_cols: list) -> pd.DataFrame:
        out = df.copy()
        if all(col in out.columns for col in required_cols):
            return out
        if out.index.name in required_cols or any(name in required_cols for name in getattr(out.index, "names", []) if name is not None):
            out = out.reset_index()
            if all(col in out.columns for col in required_cols):
                return out
        if "index" in out.columns and "user_id" in required_cols and "user_id" not in out.columns:
            out = out.rename(columns={"index": "user_id"})
            if all(col in out.columns for col in required_cols):
                return out
        missing = [col for col in required_cols if col not in out.columns]
        raise KeyError(f"Missing required columns {missing}. Available columns: {list(out.columns)}")

    req1 = on_cols + [score_col1]
    req2 = on_cols + [score_col2]
    df1n = _normalize_merge_frame(df1, req1)
    df2n = _normalize_merge_frame(df2, req2)

    for col in on_cols:
        df1n[col] = df1n[col].astype(str)
        df2n[col] = df2n[col].astype(str)

    rank_cols = [c for c in on_cols if c != "item_id"]
    if "item_id" not in on_cols:
        raise KeyError("on_cols must include 'item_id' for ranking")

    df1r = df1n[req1].copy()
    df2r = df2n[req2].copy()
    if rank_cols:
        df1r["rank_1"] = df1r.groupby(rank_cols)[score_col1].rank(method="first", ascending=False)
        df2r["rank_2"] = df2r.groupby(rank_cols)[score_col2].rank(method="first", ascending=False)
    else:
        df1r["rank_1"] = df1r[score_col1].rank(method="first", ascending=False)
        df2r["rank_2"] = df2r[score_col2].rank(method="first", ascending=False)

    merged = pd.merge(df1r[on_cols + ["rank_1"]], df2r[on_cols + ["rank_2"]], on=on_cols, how="inner")
    merged[out_rating_col] = (1.0 / (k + merged["rank_1"])) + (1.0 / (k + merged["rank_2"]))
    return merged[on_cols + [out_rating_col]]

In [118]:
# RRF: combine SVD and regular CB
hybrid_svd_cb_rrf = combine_predictions_rrf(df1=unobserved_preds_svd, df2=cb_out, on_cols=["user_id", "item_id"], score_col1="svd_rating", score_col2="cb_rating", k=60, out_rating_col="hybrid_svd_cb_rrf")
print(f"Hybrid SVD + Regular CB (RRF) rows: {len(hybrid_svd_cb_rrf)}")
hybrid_svd_cb_rrf.head()

Hybrid SVD + Regular CB (RRF) rows: 1261123


,user_id,item_id,hybrid_svd_cb_rrf
0,AELD7NXSVVFKTNFB3I673NGJVSWA,B00J4Y6L2Y,0.020043
1,AF52HVGBBCP3JAS6B7B3I4AVIWRQ,B017W1771Y,0.017532
2,AHMG3HUA4K6H475VQGJF2QAOTFTQ,B0BDWVBWC9,0.017508
3,AF52HVGBBCP3JAS6B7B3I4AVIWRQ,B004QEV0MI,0.018252
4,AGP7FLSJ7BHYFVZY3FRKVVDBH47Q,B007CSF3GO,0.023290


In [119]:
# RRF: combine SVD and LLM CB on the candidate subset
cb_llm_subset = cb_out_llm[cb_out_llm["item_id"].isin(candidate_item_ids_svd)].copy()
hybrid_svd_cb_llm_rrf = combine_predictions_rrf(df1=topk_per_user_svd, df2=cb_llm_subset, on_cols=["user_id", "item_id"], score_col1="svd_rating", score_col2="cb_rating", k=60, out_rating_col="hybrid_svd_cb_llm_rrf")
print(f"Hybrid SVD + {model_name} LLM CB (RRF) rows: {len(hybrid_svd_cb_llm_rrf)}")
hybrid_svd_cb_llm_rrf.head()

Hybrid SVD + gemma3 LLM CB (RRF) rows: 13826


,user_id,item_id,hybrid_svd_cb_llm_rrf
0,AE25ZDXYBK3LHKCZ7XUODANPME4A,B000FQ9QVI,0.018382
1,AE25ZDXYBK3LHKCZ7XUODANPME4A,B004HILZUU,0.026333
2,AE25ZDXYBK3LHKCZ7XUODANPME4A,B00000JRSB,0.020898
3,AE25ZDXYBK3LHKCZ7XUODANPME4A,B00001X50M,0.018515
4,AE25ZDXYBK3LHKCZ7XUODANPME4A,B0002A6CQ4,0.017949


### Evaluations

In [120]:
# reusable evaluation helper for any hybrid recommendation dataframe
def evaluate_hybrid_model(test_df: pd.DataFrame, hybrid_df: pd.DataFrame, hybrid_score_col: str, model_label: str, k: int = 10, relevance_threshold: float = 3, user_col: str = "user_id", item_col: str = "item_id", rating_col: str = "rating") -> pd.DataFrame:
    eval_df = hybrid_df.rename(columns={hybrid_score_col: "hybrid_rating"}).copy()

    hr = eh.calculate_hit_rate(
        test_df=test_df,
        recommendations_df=eval_df,
        prediction_col="hybrid_rating",
        k=k,
        relevance_threshold=relevance_threshold,
        user_col=user_col,
        item_col=item_col,
        rating_col=rating_col,
    )
    precision = eh.calculate_precision_at_k(
        test_df=test_df,
        recommendations_df=eval_df,
        prediction_col="hybrid_rating",
        k=k,
        relevance_threshold=relevance_threshold,
        user_col=user_col,
        item_col=item_col,
        rating_col=rating_col,
    )
    _, map_k = eh.calculate_map_at_k(
        test_df=test_df,
        recommendations_df=eval_df,
        prediction_col="hybrid_rating",
        k=k,
        relevance_threshold=relevance_threshold,
        user_col=user_col,
        item_col=item_col,
        rating_col=rating_col,
    )
    mrr = eh.calculate_mrr_at_k(
        test_df=test_df,
        recommendations_df=eval_df,
        prediction_col="hybrid_rating",
        k=k,
        relevance_threshold=relevance_threshold,
        user_col=user_col,
        item_col=item_col,
        rating_col=rating_col,
    )
    coverage = eh.calculate_coverage_at_k(
        test_df=test_df,
        recommendations_df=eval_df,
        prediction_col="hybrid_rating",
        k=k,
        user_col=user_col,
        item_col=item_col,
    )

    results = pd.DataFrame([
        {
            "Model": model_label,
            "HitRate@K": hr,
            "Precision@K": precision,
            "MAP@K": map_k,
            "MRR@K": mrr,
            "Coverage@K": coverage,
        }
    ])

    print(f"{model_label} - Hit Rate@{k}: {hr:.4f}")
    print(f"{model_label} - Precision@{k}: {precision:.4f}")
    print(f"{model_label} - MAP@{k}: {map_k:.4f}")
    print(f"{model_label} - MRR@{k}: {mrr:.4f}")
    print(f"{model_label} - Coverage@{k}: {coverage:.4f}")

    return results



In [121]:
# evaluate hybrid SVD + regular CB
results_hybrid_svd_cb = evaluate_hybrid_model(
    test_df=test_df,
    hybrid_df=hybrid_svd_cb,
    hybrid_score_col="hybrid_svd_cb_rating",
    model_label="Hybrid SVD + Regular CB",
    k=10,
    relevance_threshold=3,
    user_col="user_id",
    item_col="item_id",
    rating_col="rating",
)
results_hybrid_svd_cb

Hybrid SVD + Regular CB - Hit Rate@10: 0.2333
Hybrid SVD + Regular CB - Precision@10: 0.0272
Hybrid SVD + Regular CB - MAP@10: 0.0215
Hybrid SVD + Regular CB - MRR@10: 0.0762
Hybrid SVD + Regular CB - Coverage@10: 0.6321


,Model,HitRate@K,Precision@K,MAP@K,MRR@K,Coverage@K
0,Hybrid SVD + Regular CB,0.233261,0.027214,0.021489,0.076182,0.632147


In [122]:
# evaluate hybrid SVD + Gemma LLM CB using the same reusable function
results_hybrid_svd_cb_llm = evaluate_hybrid_model(
    test_df=test_df,
    hybrid_df=hybrid_svd_cb_llm,
    hybrid_score_col="hybrid_svd_cb_llm_rating",
    model_label=f"Hybrid SVD + {model_name} LLM CB",
    k=10,
    relevance_threshold=3,
    user_col="user_id",
    item_col="item_id",
    rating_col="rating",
)

# optional side-by-side summary
comparison_df = pd.concat([results_hybrid_svd_cb, results_hybrid_svd_cb_llm], ignore_index=True)
comparison_df

Hybrid SVD + gemma3 LLM CB - Hit Rate@10: 0.0540
Hybrid SVD + gemma3 LLM CB - Precision@10: 0.0056
Hybrid SVD + gemma3 LLM CB - MAP@10: 0.0061
Hybrid SVD + gemma3 LLM CB - MRR@10: 0.0241
Hybrid SVD + gemma3 LLM CB - Coverage@10: 0.5858


,Model,HitRate@K,Precision@K,MAP@K,MRR@K,Coverage@K
0,Hybrid SVD + Regular CB,0.233261,0.027214,0.021489,0.076182,0.632147
1,Hybrid SVD + gemma3 LLM CB,0.053996,0.005616,0.006094,0.024054,0.585761


In [123]:
# evaluate RRF hybrids using the same reusable function
results_hybrid_svd_cb_rrf = evaluate_hybrid_model(
    test_df=test_df,
    hybrid_df=hybrid_svd_cb_rrf,
    hybrid_score_col="hybrid_svd_cb_rrf",
    model_label="Hybrid SVD + Regular CB (RRF)",
    k=10,
    relevance_threshold=3,
    user_col="user_id",
    item_col="item_id",
    rating_col="rating",
)

results_hybrid_svd_cb_llm_rrf = evaluate_hybrid_model(
    test_df=test_df,
    hybrid_df=hybrid_svd_cb_llm_rrf,
    hybrid_score_col="hybrid_svd_cb_llm_rrf",
    model_label=f"Hybrid SVD + {model_name} LLM CB (RRF)",
    k=10,
    relevance_threshold=3,
    user_col="user_id",
    item_col="item_id",
    rating_col="rating",
)

comparison_df_all = pd.concat(
    [
        results_hybrid_svd_cb,
        results_hybrid_svd_cb_llm,
        results_hybrid_svd_cb_rrf,
        results_hybrid_svd_cb_llm_rrf,
    ],
    ignore_index=True,
)
comparison_df_all

Hybrid SVD + Regular CB (RRF) - Hit Rate@10: 0.1670
Hybrid SVD + Regular CB (RRF) - Precision@10: 0.0186
Hybrid SVD + Regular CB (RRF) - MAP@10: 0.0163
Hybrid SVD + Regular CB (RRF) - MRR@10: 0.0616
Hybrid SVD + Regular CB (RRF) - Coverage@10: 0.5933
Hybrid SVD + gemma3 LLM CB (RRF) - Hit Rate@10: 0.0540
Hybrid SVD + gemma3 LLM CB (RRF) - Precision@10: 0.0056
Hybrid SVD + gemma3 LLM CB (RRF) - MAP@10: 0.0060
Hybrid SVD + gemma3 LLM CB (RRF) - MRR@10: 0.0245
Hybrid SVD + gemma3 LLM CB (RRF) - Coverage@10: 0.5858


,Model,HitRate@K,Precision@K,MAP@K,MRR@K,Coverage@K
0,Hybrid SVD + Regular CB,0.233261,0.027214,0.021489,0.076182,0.632147
1,Hybrid SVD + gemma3 LLM CB,0.053996,0.005616,0.006094,0.024054,0.585761
2,Hybrid SVD + Regular CB (RRF),0.167027,0.018647,0.016285,0.061578,0.593312
3,Hybrid SVD + gemma3 LLM CB (RRF),0.053996,0.005616,0.005996,0.024483,0.585761


LLM CB preforms very poorly as a stand alone model compared to regular CB

Compare gemma against qwen each other, 1.2 billion model vs a 4 billion parameter model.

The fact that I use the same canidate set for both gemma3 and qwen3 means both LLMs just re rank the same set.
The prompt template and temperature set to 0 means the stylistic differences are pretty much the same for the descriptions.
the text normaliztion also removes some this aspect as well but its needed. 
My tf-idf vector space conversion just uses primative average for user vectors. 


I don't tune or grid search like in week 10 which i think is the biggest difference. I had added catergories and features in the description itself but it actually reduced the performance.

It also appears as tho combining cf with llm cb reduces performance this could be due the weighted combination method and the alpha beta values
